# ACS in-hospital mortality prediction

This notebook evaluates a 13-feature Random Forest in 1,524 patients with acute coronary syndrome. The primary validation uses five stratified folds repeated with ten fixed seeds. All preprocessing is estimated within each training fold.

The patient-level mean of the ten out-of-fold predictions is used only for threshold selection and the paired GRACE 2.0 comparison. Seed-level AUC variation is retained for uncertainty reporting.

## TRIPOD+AI reporting checklist

| Item | Implementation |
|---|---|
| Population and outcome | ACS, Killip I to III, in-hospital mortality |
| Sample size | 1,524 patients, 115 deaths |
| Predictors | 13 admission variables specified before analysis |
| Missing data | Training-fold median imputation |
| Internal validation | Five folds repeated over ten fixed seeds |
| Performance | AUC, 95% CI, range, AUPRC, Brier score, threshold metrics |
| Comparator | Patient-level GRACE 2.0 score from eight source variables |
| Model interpretation | Mean Gini importance over all 50 validation models |
| Reproducibility | Versions, seeds, filters, and schema reported below |

## 1. Execute the canonical analysis

The standalone pipeline is the single source of numerical results. Running it here prevents notebook outputs from drifting away from the script, figures, or thesis tables.

In [1]:
import subprocess, sys
run = subprocess.run([sys.executable, "thesis_main.py"], check=True, text=True, capture_output=True)
print(run.stdout)

Cohort: N=1524, deaths=115 (7.5%)
Seed 42: AUC=0.8172
Seed 123: AUC=0.8157
Seed 456: AUC=0.8247
Seed 789: AUC=0.8024
Seed 111: AUC=0.8086
Seed 222: AUC=0.8220
Seed 333: AUC=0.8144
Seed 444: AUC=0.8237
Seed 555: AUC=0.8210
Seed 666: AUC=0.8074
AUC=0.8157 ± 0.0075, 95% CI 0.8110-0.8204, range 0.8024-0.8247
Brier=0.0605; AUPRC=0.3005
Safety=0.018455: sens=98.3%, spec=26.2%, FN=2
Youden=0.103981: sens=71.3%, spec=82.0%
GRACE AUC=0.7767; RF ensemble AUC=0.8189; McNemar p=4.908e-10
Triage: {'Ward': {'n': 371, 'death': 2, 'rate': 0.005390835579514825}, 'HCU': {'n': 817, 'death': 31, 'rate': 0.037943696450428395}, 'ICU': {'n': 336, 'death': 82, 'rate': 0.24404761904761904}}



The execution log reports one AUC for each prespecified seed. Its final lines report the seed-level summary, exact thresholds, paired GRACE comparison, and mutually exclusive triage counts.

## 2. Load the synchronized results

The JSON artifact contains full-precision values. Display rounding is applied only when results are presented.

In [2]:
import json
from pathlib import Path
r = json.loads(Path("validation_results.json").read_text())
d, m, t, g = r["dataset"], r["metrics"], r["thresholds"], r["grace"]
print(f'N={d["n"]:,}; deaths={d["deaths"]} ({d["prevalence"]:.1%})')
print(f'Seed AUC={m["auc_mean"]:.4f} ± {m["auc_sd"]:.4f}; 95% CI {m["auc_ci95"][0]:.4f} to {m["auc_ci95"][1]:.4f}')
print(f'Range={m["auc_min"]:.4f} to {m["auc_max"]:.4f}; mean-OOF AUC={m["ensemble_auc"]:.4f}')
print(f'Brier={m["brier_mean"]:.4f}; AUPRC={m["auprc_mean"]:.4f}')

N=1,524; deaths=115 (7.5%)
Seed AUC=0.8157 ± 0.0075; 95% CI 0.8110 to 0.8204
Range=0.8024 to 0.8247; mean-OOF AUC=0.8189
Brier=0.0605; AUPRC=0.3005


The mean seed AUC quantifies performance across repeated partitions. The mean-OOF AUC is slightly higher because averaging ten patient-level predictions reduces prediction noise. The Brier score and AUPRC replace the stale values previously shown in this notebook.

## 3. Baseline characteristics

Continuous variables use Welch's t-test. Binary rows use a 2 by 2 chi-square or Fisher exact test. The separate Killip helper uses the full 2 by 3 table, so Killip classes I, II, and III are retained in the multicategory test.

In [3]:
import pandas as pd
baseline = pd.DataFrame(r["baseline"])
baseline["p"] = baseline["p"].map(lambda x: f"{x:.4f}")
display(baseline)
print(f'Killip I to III multicategory p={r["killip_multicategory_p"]:.6g}')

                variable         alive          death       p
0            Age (years)   56.9 ± 11.0     64.5 ± 9.2  0.0000
1            Male gender  1145 (81.3%)     83 (72.2%)  0.0247
2                  STEMI   975 (69.2%)     72 (62.6%)  0.1736
3       Heart rate (bpm)   82.5 ± 19.0    91.0 ± 20.2  0.0000
4             SBP (mmHg)  132.1 ± 24.2   127.3 ± 28.6  0.0827
5              RR (/min)    21.6 ± 3.7     23.7 ± 4.0  0.0000
6             Killip III    108 (7.7%)     37 (32.2%)  0.0000
7      Hemoglobin (g/dL)    13.8 ± 2.0     12.4 ± 2.4  0.0000
8          Ureum (mg/dL)   38.3 ± 26.3    71.8 ± 53.3  0.0000
9   eGFR (mL/min/1.73m2)   83.4 ± 26.0    57.8 ± 29.0  0.0000
10     Potassium (mEq/L)     4.1 ± 0.6      4.4 ± 1.0  0.0009
11            APTT (sec)   27.7 ± 12.9    31.6 ± 20.0  0.0438
12           GDS (mg/dL)  179.9 ± 90.3  198.8 ± 116.9  0.0952
13              LVEF (%)    42.8 ± 7.7     38.1 ± 8.8  0.0000
14         LVOT VTI (cm)    17.7 ± 4.7     15.5 ± 5.6  0.0001
15      

Deaths were associated with an older and physiologically higher-risk profile across several admission variables. The binary Killip III row remains suitable for the clinical baseline table, while the full multicategory test evaluates the complete Killip distribution.

## 4. Threshold selection and clinical triage

The safety threshold is the largest observed cutoff that permits no more than two false negatives among 115 deaths. The Youden threshold maximizes sensitivity plus specificity minus one. Both thresholds are derived from the patient-level mean OOF vector.

In [4]:
s, y = t["safety_metrics"], t["youden_metrics"]
print(f'Safety threshold={t["safety"]:.6f}: sensitivity={s["sensitivity"]:.1%}, specificity={s["specificity"]:.1%}, FN={s["fn"]}, FP={s["fp"]}')
print(f'Youden threshold={t["youden"]:.6f}: sensitivity={y["sensitivity"]:.1%}, specificity={y["specificity"]:.1%}, FN={y["fn"]}, FP={y["fp"]}')
triage = pd.DataFrame(t["triage"]).T
triage["mortality"] = triage["rate"].map(lambda x: f"{x:.1%}")
display(triage[["n", "death", "mortality"]])

Safety threshold=0.018455: sensitivity=98.3%, specificity=26.2%, FN=2, FP=1040
Youden threshold=0.103981: sensitivity=71.3%, specificity=82.0%, FN=33, FP=254
          n  death mortality
Ward  371.0    2.0      0.5%
HCU   817.0   31.0      3.8%
ICU   336.0   82.0     24.4%


The safety rule assigns 371 patients to Ward and misses two deaths. The Youden cutoff separates 336 ICU patients with 24.4% observed mortality. These are validation-derived decision rules and require external assessment before clinical deployment.

## 5. GRACE 2.0 comparison

GRACE points are calculated from age, heart rate, systolic pressure, creatinine, Killip class, arrest at admission, elevated troponin, and ST deviation. Arrest is mapped to `rosc_in_igd`, elevated enzymes to troponin above 0.04 ng/mL, and ST deviation to `ekg_ste`. Six missing creatinine values use the cohort median for this fixed-score comparator.

In [5]:
q = g["threshold_20pct"]
comparison = pd.DataFrame({"Random Forest": q["rf"], "GRACE 2.0": q["grace"]})
display(comparison.loc[["sensitivity", "specificity", "ppv", "npv", "tn", "fp", "fn", "tp"]])
print(f'RF mean-OOF AUC={g["rf_auc"]:.4f}; GRACE AUC={g["auc"]:.4f}; delta={g["auc_delta"]:.4f}')
print(f'Bootstrap delta 95% CI={g["auc_delta_ci95"][0]:.4f} to {g["auc_delta_ci95"][1]:.4f}; p={g["auc_p_bootstrap"]:.4g}')
print(f'GRACE 20% risk begins at score {q["grace_score_min"]}; McNemar p={q["mcnemar_p"]:.4g}')

             Random Forest    GRACE 2.0
sensitivity       0.408696     0.521739
specificity       0.928318     0.853087
ppv               0.317568     0.224719
npv               0.950581     0.956245
tn             1308.000000  1202.000000
fp              101.000000   207.000000
fn               68.000000    55.000000
tp               47.000000    60.000000
RF mean-OOF AUC=0.8189; GRACE AUC=0.7767; delta=0.0422
Bootstrap delta 95% CI=0.0034 to 0.0839; p=0.029
GRACE 20% risk begins at score 163; McNemar p=4.908e-10


The Random Forest mean-OOF vector discriminates mortality better than the reconstructed GRACE score in this cohort. At a 20% predicted-risk cutoff, the paired McNemar test evaluates whether patient-level classification errors differ between methods.

## 6. Cross-validated feature importance

Importance is aggregated over the same 50 fitted fold models used for validation. No full-cohort refit contributes to this table.

In [6]:
importance = pd.DataFrame(r["feature_importance"]).T.reset_index(names="feature")
display(importance.style.format({"mean": "{:.4f}", "sd": "{:.4f}"}))

Renal function and age have the largest mean impurity reductions. Gini importance ranks model usage rather than causal effect, and correlated predictors can share importance.

## 7. Reproducibility record

The following cell prints package versions, exact split seeds, cohort filter, and feature schema. The parquet file is read-only throughout the workflow.

In [7]:
print('Versions:', r["versions"])
print('Seeds:', r["seeds"])
print('Cohort filter:', r["dataset"]["filter"])
print('Feature schema:', r["features"])
print('Threshold method:', r["thresholds"]["method"])

Versions: {'python': '3.11.11', 'pandas': '3.0.3', 'numpy': '2.4.6', 'scikit_learn': '1.9.0', 'scipy': '1.17.1'}
Seeds: [42, 123, 456, 789, 111, 222, 333, 444, 555, 666]
Cohort filter: pat_exclude=False and Killip I-III
Feature schema: ['age_when_admission', 'ureum_igd', 'egfr_igd', 'hr', 'hb_igd', 'killip', 'sbp', 'rr', 'lvef', 'lvot_vti_igd', 'tapse_value', 'kalium_igd', 'aptt_value']
Threshold method: Thresholds derived from the patient-level mean of 10 repeated OOF predictions. Safety is the maximum observed cutoff yielding FN <= 2; Youden maximizes sensitivity + specificity - 1.


The recorded environment and fixed seeds support exact reruns in the current repository. The schema lists all model inputs and distinguishes the modeling features from the separate GRACE variables.

## 8. Final synchronized metrics

This compact table is the reference for the thesis narrative and validation script.

In [8]:
summary = pd.DataFrame([
    ["Patients", d["n"]], ["Deaths", d["deaths"]],
    ["Mean seed AUC", f'{m["auc_mean"]:.4f}'], ["Mean-OOF AUC", f'{m["ensemble_auc"]:.4f}'],
    ["Brier", f'{m["brier_mean"]:.4f}'], ["AUPRC", f'{m["auprc_mean"]:.4f}'],
    ["Safety threshold", f'{t["safety"]:.6f}'], ["Youden threshold", f'{t["youden"]:.6f}'],
    ["GRACE AUC", f'{g["auc"]:.4f}'], ["McNemar p", f'{g["threshold_20pct"]["mcnemar_p"]:.4g}']])
display(summary.rename(columns={0: "Metric", 1: "Value"}))

             Metric      Value
0          Patients       1524
1            Deaths        115
2     Mean seed AUC     0.8157
3      Mean-OOF AUC     0.8189
4             Brier     0.0605
5             AUPRC     0.3005
6  Safety threshold   0.018455
7  Youden threshold   0.103981
8         GRACE AUC     0.7767
9         McNemar p  4.908e-10


All values in this summary are generated from the corrected pipeline. The thesis headline can state an OOF AUC of approximately 0.818, while the methods and results must also report the seed-level mean, uncertainty interval, and range shown above.